# Clean floor-plan training
Use the `.venv-training` Python kernel. Existing runs are preserved. Do not run training here while the background run is active.

Labels use one semantic polygon per object in the F1_scaled frame. Outdoor spaces are excluded. Door boxes represent the opening, not the swing arc. The sanity dataset deliberately reuses training images for validation and is never a reported benchmark.


In [ ]:
from pathlib import Path
import sys
import torch
BASE = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "datasets/blueprints_clean_v2/data.yaml").exists())
print(sys.executable)
assert torch.cuda.is_available(), "Select .venv-training kernel"
print(torch.cuda.get_device_name(0))


In [ ]:
import json
audit = json.loads((BASE / "datasets/blueprints_clean_v2/audit.json").read_text())
for split, info in audit["splits"].items():
    print(split, info["images"], info["boxes"], "rejected:", len(info["rejected"]))


In [ ]:
from IPython.display import display, Image
previews = sorted((BASE / "datasets/blueprints_clean_v2/previews").glob("val_*.jpg"))
for p in previews[:3]:
    display(Image(filename=str(p), width=1000))


In [ ]:
sys.path.insert(0, str(BASE / "vision"))
from train_blueprints import train_blueprint_model
# Optional diagnostic, not held-out accuracy:
# train_blueprint_model(epochs=40, sanity=True)
model = train_blueprint_model(epochs=80)


## Final held-out evaluation
After selecting settings using validation only, evaluate the selected best.pt on the official test split once. Do not tune on the test results.


In [ ]:
from ultralytics import YOLO
# Set this to the actual new run printed by training.
RUN = BASE / "runs/blueprint_clean_gpu"
assert (RUN / "weights/best.pt").exists()
best = YOLO(str(RUN / "weights/best.pt"))
# Uncomment when model selection is complete:
# metrics = best.val(data=str(BASE / "datasets/blueprints_clean_v2/data.yaml"), split="test", imgsz=960, batch=1, device=0, workers=0, max_det=600, project=str(BASE / "runs"), name="blueprint_clean_test")
